In [ ]:
import os
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data"

print("Project root:", PROJECT_ROOT)
print("Data dir exists:", DATA_DIR.exists())
print("Files in data:")
for p in sorted(DATA_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(PROJECT_ROOT))

In [ ]:
import pandas as pd
from pathlib import Path

base_files = [
    "application_train.csv",
    "bureau.csv",
    "previous_application.csv",
    "installments_payments.csv",
]

for fname in base_files:
    path = DATA_DIR / fname
    df = pd.read_csv(path, nrows=5)
    full = pd.read_csv(path, usecols=[0])  # cheap row count
    print("=" * 80)
    print(fname)
    print("Rows:", len(pd.read_csv(path)))
    print("Columns:", len(df.columns))
    print("First columns:", list(df.columns[:15]))

In [ ]:
key_checks = {
    "application_train.csv": ["SK_ID_CURR", "TARGET"],
    "bureau.csv": ["SK_ID_CURR", "SK_ID_BUREAU"],
    "previous_application.csv": ["SK_ID_CURR", "SK_ID_PREV"],
    "installments_payments.csv": ["SK_ID_CURR", "SK_ID_PREV"],
}

for fname, cols in key_checks.items():
    path = DATA_DIR / fname
    sample = pd.read_csv(path, nrows=5)
    print("=" * 80)
    print(fname)
    for c in cols:
        print(c, "->", c in sample.columns)

In [ ]:
import duckdb
from pathlib import Path

DB_PATH = PROJECT_ROOT / "credit_risk.duckdb"
con = duckdb.connect(str(DB_PATH))

print("DB path:", DB_PATH)
print("Exists:", DB_PATH.exists())

In [ ]:
con.execute(f"""
CREATE OR REPLACE VIEW application_train AS
SELECT * FROM read_csv_auto('{(DATA_DIR / "application_train.csv").as_posix()}');

CREATE OR REPLACE VIEW bureau AS
SELECT * FROM read_csv_auto('{(DATA_DIR / "bureau.csv").as_posix()}');

CREATE OR REPLACE VIEW previous_application AS
SELECT * FROM read_csv_auto('{(DATA_DIR / "previous_application.csv").as_posix()}');

CREATE OR REPLACE VIEW installments_payments AS
SELECT * FROM read_csv_auto('{(DATA_DIR / "installments_payments.csv").as_posix()}');
""")

print("Views created.")

In [ ]:
queries = {
    "application_train_count": "SELECT COUNT(*) AS n FROM application_train",
    "bureau_count": "SELECT COUNT(*) AS n FROM bureau",
    "previous_application_count": "SELECT COUNT(*) AS n FROM previous_application",
    "installments_payments_count": "SELECT COUNT(*) AS n FROM installments_payments",
}

for name, q in queries.items():
    print("=" * 80)
    print(name)
    print(con.execute(q).df())

In [ ]:
for table in ["application_train", "bureau", "previous_application", "installments_payments"]:
    print("=" * 100)
    print(table)
    display(con.execute(f"DESCRIBE {table}").df())

In [ ]:
checks = {
    "application_train_unique_applicants": """
        SELECT 
            COUNT(*) AS rows,
            COUNT(DISTINCT SK_ID_CURR) AS distinct_curr
        FROM application_train
    """,
    "bureau_applicant_grain": """
        SELECT 
            COUNT(*) AS rows,
            COUNT(DISTINCT SK_ID_CURR) AS distinct_curr,
            COUNT(DISTINCT SK_ID_BUREAU) AS distinct_bureau
        FROM bureau
    """,
    "previous_application_grain": """
        SELECT 
            COUNT(*) AS rows,
            COUNT(DISTINCT SK_ID_CURR) AS distinct_curr,
            COUNT(DISTINCT SK_ID_PREV) AS distinct_prev
        FROM previous_application
    """,
    "installments_grain": """
        SELECT 
            COUNT(*) AS rows,
            COUNT(DISTINCT SK_ID_CURR) AS distinct_curr,
            COUNT(DISTINCT SK_ID_PREV) AS distinct_prev
        FROM installments_payments
    """,
}

for name, q in checks.items():
    print("=" * 80)
    print(name)
    display(con.execute(q).df())

In [ ]:
from pathlib import Path

report_path = PROJECT_ROOT / "outputs" / "tables" / "schema_grain_report.txt"

tables = ["application_train", "bureau", "previous_application", "installments_payments"]

checks = {
    "application_train_unique_applicants": """
        SELECT COUNT(*) AS rows,
               COUNT(DISTINCT SK_ID_CURR) AS distinct_curr
        FROM application_train
    """,
    "bureau_applicant_grain": """
        SELECT COUNT(*) AS rows,
               COUNT(DISTINCT SK_ID_CURR) AS distinct_curr,
               COUNT(DISTINCT SK_ID_BUREAU) AS distinct_bureau
        FROM bureau
    """,
    "previous_application_grain": """
        SELECT COUNT(*) AS rows,
               COUNT(DISTINCT SK_ID_CURR) AS distinct_curr,
               COUNT(DISTINCT SK_ID_PREV) AS distinct_prev
        FROM previous_application
    """,
    "installments_grain": """
        SELECT COUNT(*) AS rows,
               COUNT(DISTINCT SK_ID_CURR) AS distinct_curr,
               COUNT(DISTINCT SK_ID_PREV) AS distinct_prev
        FROM installments_payments
    """
}

with open(report_path, "w", encoding="utf-8") as f:
    f.write("CREDIT RISK INTELLIGENCE — SCHEMA + GRAIN REPORT\n")
    f.write("=" * 100 + "\n\n")

    for table in tables:
        f.write(f"TABLE: {table}\n")
        f.write("-" * 100 + "\n")
        desc_df = con.execute(f"DESCRIBE {table}").df()
        f.write(desc_df.to_string(index=False))
        f.write("\n\n")

    f.write("=" * 100 + "\n")
    f.write("GRAIN CHECKS\n")
    f.write("=" * 100 + "\n\n")

    for name, query in checks.items():
        f.write(f"CHECK: {name}\n")
        f.write("-" * 100 + "\n")
        out_df = con.execute(query).df()
        f.write(out_df.to_string(index=False))
        f.write("\n\n")

print(f"Saved report to: {report_path}")

In [ ]:
import duckdb
from pathlib import Path

DB_PATH = PROJECT_ROOT / "credit_risk.duckdb"
con = duckdb.connect(str(DB_PATH))

print("DB path:", DB_PATH)
print("Exists:", DB_PATH.exists())
sql_path = PROJECT_ROOT / "sql" / "03_bureau_agg.sql"
bureau_sql = sql_path.read_text(encoding="utf-8")
con.execute(bureau_sql)
print("bureau_agg created")